# Discord Persona Fine-Tuning: Mistral NeMo 12B

This notebook trains a QLoRA adapter on top of Mistral NeMo 12B to clone a specific Discord user's conversational style. It is heavily optimized to run on a free Google Colab T4 GPU (15GB VRAM).

### Key Optimizations Enabled:
- **4-bit Quantization (NF4):** Shrinks the 12B model down to ~6.5GB.
- **Gradient Checkpointing:** Trades computation time for massive VRAM savings.
- **Paged 8-bit Optimizer:** Pages optimizer states to system RAM to prevent OOM spikes.
- **Sequence Length Capping:** Limits context to 2048 tokens to keep attention matrices small.
- **AES-256 Encryption Support:** Protects your dataset and resulting model weights.

In [ ]:
# ==========================================
# USER CONFIGURATION PARAMETERS
# ==========================================

# 1. Dataset Source Configuration
# If your zip is already in your Google Drive, provide the path here (fastest/safest method)
LOCAL_DRIVE_ZIP_PATH = '' # e.g., '/content/drive/MyDrive/dataset.zip'

# Otherwise, set the shared Google Drive link for your dataset zip here.
# The zip MUST contain the file structure: processed/dataset.jsonl
GDRIVE_SHARED_LINK = 'DRIVE_LINK_HERE'

# 1.5 AES-256 Encryption / Decryption Keys
# If your dataset is an AES-256 encrypted zip, provide the password here.
DECRYPTION_KEY = ""
# The output model will be zipped and encrypted with this key.
# If blank, it defaults to DECRYPTION_KEY. If both are blank, it saves unencrypted.
ENCRYPTION_KEY = ""

# 2. Seed Control
USER_SEED = 42

# 3. Training Hyperparameters (T4 Optimized)
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1           # Keep at 1 to prevent OOM
GRAD_ACCUM_STEPS = 4     # Simulates a batch size of 4
EPOCHS = 1               # 1 to 3 is usually enough for style cloning
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

In [ ]:
import os
import datetime
import time
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Setup Timestamped Directory
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_dir = f"/content/drive/MyDrive/soulclone/{timestamp}"
os.makedirs(run_output_dir, exist_ok=True)

# Global Logging Wrapper
master_log = ""
def log_print(*args, **kwargs):
    global master_log
    output = " ".join(map(str, args))
    print(output, **kwargs)
    master_log += output + "\n"

# Timing Wrapper
section_times = {}
global_start_time = time.time()

def start_timer(section_name):
    section_times[section_name] = {'start_time': time.time()}
    start_str = datetime.datetime.fromtimestamp(section_times[section_name]['start_time']).strftime('%H:%M:%S')
    log_print(f"\n[INFO] Started '{section_name}' at {start_str}")

def stop_timer(section_name):
    if section_name in section_times and 'start_time' in section_times[section_name]:
        end_t = time.time()
        section_times[section_name]['end_time'] = end_t
        duration = end_t - section_times[section_name]['start_time']
        section_times[section_name]['duration'] = duration
        end_str = datetime.datetime.fromtimestamp(end_t).strftime('%H:%M:%S')
        log_print(f"[INFO] Finished '{section_name}' at {end_str} (Took {duration:.2f}s)")

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
log_print("\n" + "="*50)
log_print(f"RUN STARTED: {current_time}")
log_print(f"CONFIGURATION (LOCAL PATH): {LOCAL_DRIVE_ZIP_PATH}")
log_print(f"CONFIGURATION (DRIVE LINK): {GDRIVE_SHARED_LINK}")
log_print(f"AES ENCRYPTION ENABLED: {'Yes' if (ENCRYPTION_KEY or DECRYPTION_KEY) else 'No'}")
log_print(f"SEED: {USER_SEED}")
log_print(f"OUTPUT DIR: {run_output_dir}")
log_print("="*50 + "\n")

In [ ]:
start_timer("1. Install Dependencies")
!pip install -q transformers datasets peft trl bitsandbytes gdown accelerate pyzipper
stop_timer("1. Install Dependencies")

In [ ]:
start_timer("2. Data Download & Extraction")
import gdown
import pyzipper
import shutil

local_zip = '/content/dataset.zip'
local_extract_dir = '/content/local_data'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# Fetch dataset file
if LOCAL_DRIVE_ZIP_PATH and os.path.exists(LOCAL_DRIVE_ZIP_PATH):
    log_print(f"Copying dataset locally from: {LOCAL_DRIVE_ZIP_PATH}")
    shutil.copy(LOCAL_DRIVE_ZIP_PATH, local_zip)
elif GDRIVE_SHARED_LINK != 'DRIVE_LINK_HERE':
    log_print("Downloading zip from Google Drive via gdown...")
    file_id = get_gdrive_id(GDRIVE_SHARED_LINK)
    download_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(download_url, local_zip, quiet=False)
else:
    log_print("WARNING: No valid dataset source provided.")

# Extract dataset file
if os.path.exists(local_zip):
    os.makedirs(local_extract_dir, exist_ok=True)
    log_print("Extracting dataset...")
    try:
        if DECRYPTION_KEY:
            log_print("Using provided DECRYPTION_KEY for AES-256 decryption...")
            with pyzipper.AESZipFile(local_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                zip_ref.extractall(local_extract_dir)
        else:
            with pyzipper.AESZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(local_extract_dir)
    except Exception as e:
        log_print(f"Extraction failed (Check if key is missing or incorrect): {e}")

dataset_path = os.path.join(local_extract_dir, 'processed', 'dataset.jsonl')
if os.path.exists(dataset_path):
    log_print(f"Successfully located dataset at: {dataset_path}")
else:
    log_print(f"ERROR: Could not find dataset at {dataset_path}. Check your zip structure.")

stop_timer("2. Data Download & Extraction")

In [ ]:
start_timer("3. Load Model and Tokenizer")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "mistralai/Mistral-Nemo-Instruct-2407"

log_print(f"Configuring 4-bit quantization for {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # T4 safe
)

log_print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, fix_mistral_regex=True)
tokenizer.pad_token = tokenizer.eos_token

log_print("Loading Model (this will take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

model.config.pad_token_id = tokenizer.pad_token_id # Prevents training warning spam
model.config.use_cache = False # Required for gradient checkpointing
stop_timer("3. Load Model and Tokenizer")

In [ ]:
start_timer("4. Prepare Dataset")
from datasets import load_dataset

log_print("Loading JSONL dataset into HuggingFace format...")
raw_dataset = load_dataset('json', data_files=dataset_path, split='train')

def apply_chat_template(examples):
    texts = []
    for msgs in examples['messages']:
        # Automatically maps your system/user/assistant tags to NeMo's native format
        formatted_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted_text)
    return {'text': texts}

log_print("Applying Mistral NeMo chat templates to data...")
train_dataset = raw_dataset.map(apply_chat_template, batched=True)
log_print(f"Prepared {len(train_dataset)} conversational examples.")
stop_timer("4. Prepare Dataset")

In [ ]:
start_timer("5. LoRA Configuration & Training")
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

# Prepare model for memory-efficient training
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

class LoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_print(f"Step {state.global_step}: {logs}")

training_args = TrainingArguments(
    output_dir=os.path.join(run_output_dir, "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    seed=USER_SEED
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    callbacks=[LoggingCallback()]
)

log_print("Starting Training...")
trainer.train()
stop_timer("5. LoRA Configuration & Training")

In [ ]:
start_timer("6. Saving & Exporting")
import matplotlib.pyplot as plt
import pyzipper

# 1. Save Final Model Weights
final_model_path = os.path.join(run_output_dir, "final_adapter")
trainer.save_model(final_model_path)
log_print(f"Saved final LoRA adapter to: {final_model_path}")

# 1.5 Encrypt Output Model (if requested)
target_enc_key = ENCRYPTION_KEY if ENCRYPTION_KEY else DECRYPTION_KEY
if target_enc_key:
    log_print("Zipping and AES-256 encrypting the final model...")
    encrypted_zip_path = os.path.join(run_output_dir, "final_adapter_encrypted.zip")
    with pyzipper.AESZipFile(encrypted_zip_path, 'w', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(target_enc_key.encode('utf-8'))
        for root, _, files in os.walk(final_model_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=os.path.dirname(final_model_path))
                zf.write(file_path, arcname)
    log_print(f"Encrypted model saved to: {encrypted_zip_path}")
else:
    log_print("No encryption key provided. Skipping output encryption.")

# 2. Graph Training Loss
history = trainer.state.log_history
loss_steps = [x['step'] for x in history if 'loss' in x]
loss_values = [x['loss'] for x in history if 'loss' in x]

if loss_values:
    plt.figure(figsize=(10, 5))
    plt.plot(loss_steps, loss_values, label='Training Loss', color='blue')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training Loss over Time')
    plt.legend()
    plt.grid(True)
    loss_graph_path = os.path.join(run_output_dir, f"training_loss_{timestamp}.png")
    plt.savefig(loss_graph_path, bbox_inches='tight')
    plt.show()
    log_print(f"Saved loss graph to: {loss_graph_path}")

# 3. Generate Summary Image
total_duration = time.time() - global_start_time
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

y_pos = 0.95
ax.text(0.05, y_pos, "Mistral NeMo 12B - Fine-Tuning Summary", fontsize=18, fontweight='bold')
y_pos -= 0.1
ax.text(0.05, y_pos, "Model Configuration:", fontsize=14, fontweight='bold')
y_pos -= 0.05
ax.text(0.1, y_pos, f"Base Model: mistralai/Mistral-Nemo-Instruct-2407", fontsize=12)
y_pos -= 0.05
ax.text(0.1, y_pos, f"LoRA Rank/Alpha: {LORA_R} / {LORA_ALPHA}", fontsize=12)
y_pos -= 0.05
ax.text(0.1, y_pos, f"Seed: {USER_SEED}", fontsize=12)
y_pos -= 0.1

ax.text(0.05, y_pos, "Execution Times:", fontsize=14, fontweight='bold')
y_pos -= 0.05
for section, times in section_times.items():
    if 'duration' in times:
        ax.text(0.1, y_pos, f"{section}: {times['duration']:.2f}s", fontsize=10, fontfamily='monospace')
        y_pos -= 0.04

y_pos -= 0.05
ax.text(0.05, y_pos, f"Total Execution Time: {total_duration:.2f}s", fontsize=14, fontweight='bold', color='darkgreen')

summary_img_path = os.path.join(run_output_dir, f"summary_stats_{timestamp}.png")
plt.savefig(summary_img_path, bbox_inches='tight', facecolor='white')
plt.show()
log_print(f"Saved summary image to {summary_img_path}")

# 4. Save Final Master Log
log_file_path = os.path.join(run_output_dir, f'compiled_training_log_{timestamp}.txt')
with open(log_file_path, 'w', encoding='utf-8') as f:
    f.write(master_log)
print(f"\nCompiled text log successfully saved to {log_file_path}")
stop_timer("6. Saving & Exporting")